# Set-up

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report, f1_score
import matplotlib.pyplot as plt
import importlib
import lab1_utils


#importlib.reload(lab1_utils)
#from lab1_utils import train_model, evaluate_model, plot_history, train_val_test_split

In [6]:
import wandb
wandb.login()

True

In [7]:
RANDOM_SEED = 17

if torch.cuda.is_available(): 
    device = torch.device('cuda')
    print("Device set to CUDA")
else:
    device = torch.device("cpu")
    print("Device set to CPU")

Device set to CUDA


# Utils

In [8]:
def train_model(model, train_loader, val_loader, epochs, criterion, 
                optimizer, scheduler, model_name, patience, 
                device, wandb_logging=True):

    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):

        # --- Train ---
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad() # Clear gradients
            outputs = model(X_batch) # Forward pass
            loss = criterion(outputs, y_batch) # Compute loss
            loss.backward() # Backpropagation
            optimizer.step() # Update weights

            train_loss += loss.item() * len(y_batch) # Accumulate total loss for the epoch
            correct += (outputs.argmax(1) == y_batch).sum().item() # Count correct predictions
            total += len(y_batch) # Count total samples

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad(): 
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)

                outputs = model(X_batch) # Forward pass
                loss = criterion(outputs, y_batch) # Compute loss

                val_loss += loss.item() * len(y_batch)
                val_correct += (outputs.argmax(1) == y_batch).sum().item()
                val_total += len(y_batch)

        scheduler.step() # Update learning rate

        t_loss = train_loss / total # total loss divided by total samples for average loss per sample per epoch
        v_loss = val_loss / val_total  
        t_acc  = correct / total
        v_acc  = val_correct / val_total

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        # --- WandB Logging ---
        if wandb_logging and wandb.run is not None:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": t_loss,
                "val_loss": v_loss,
                "train_acc": t_acc,
                "val_acc": v_acc,
                "lr": scheduler.get_last_lr()[0]
            })

        # --- Epoch summary ---
        print(f'[{model_name}] Epoch {epoch+1:3d}/{epochs} | '
                f'Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | '
                f'Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}')

        # --- Early Stopping ---
        if v_loss < best_val_loss- 1e-4: # Consider a small threshold to avoid saving on negligible improvements
            best_val_loss = v_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[{model_name}] Early stopping at epoch {epoch+1}")
                # Save the best model state to disk 
                torch.save(best_state, f'{model_name}_best.pth')
                break
    
    if best_state is not None:
        model.load_state_dict(best_state) # Load best model state before returning

    return history


def evaluate_model(model, test_loader, model_name, device, wandb_logging=True):

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch).argmax(1).detach().cpu().numpy().tolist()
            labels = y_batch.detach().cpu().numpy().tolist()

            all_preds.extend(preds)
            all_labels.extend(labels)
            
    print(f'\n===== {model_name} — Test Results =====')
    results = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "confusion_matrix": confusion_matrix(all_labels, all_preds)
    }

    for metric, value in results.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}:\n{value}")
    
    # --- WandB Logging ---
    if wandb_logging and wandb.run is not None:        
        wandb.log({
            "test_accuracy": results['accuracy'],
            "test_precision": results['precision'],
            "test_recall": results['recall'],
            "test_f1": results['f1'],
        })
                                
    return results


def plot_history(history, title='Training History'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], label='Train Loss')
    ax1.plot(epochs, history['val_loss'], label='Val Loss')
    ax1.set_title(f'{title} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

    ax2.plot(epochs, history['train_acc'], label='Train Acc')
    ax2.plot(epochs, history['val_acc'], label='Val Acc')
    ax2.set_title(f'{title} — Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()

    plt.tight_layout()
    plt.show()

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

def preprocess_pandas(data, columns):
    """
    Preprocess text data in the 'Sentence' column of the input DataFrame.
        1. Converts all text to lowercase.
        2. Removes email addresses using a regular expression.
        3. Removes IP addresses using a regular expression.
        4. Removes special characters, keeping only word characters and whitespace.
        5. Removes digits from the text.
        6. Tokenizes the sentences and removes English stopwords.
        7. Stores the cleaned sentences in a new DataFrame with the specified columns.
    """

    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','', regex=True)                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence']) 
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return df_ # Fixed. The original function returned data instead of the cleaned dataframe df_.

def train_val_test_split(data, labels, train_size, val_size, test_size, seed):
    """
    Splits the input DataFrame into training, validation, and test sets based on the specified sizes.
    The function first shuffles the data and then splits it according to the provided ratios.
    """
    assert train_size + val_size + test_size == 1.0, "Train, validation and test sizes must sum to 1."
    
    # Split data into TRAIN +TEST
    X, X_test, y, y_test = train_test_split(
        data, labels, test_size=test_size, random_state=seed, shuffle=True
    )

    # Then TRAIN into TRAIN + VAL
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=val_size, random_state=seed, shuffle=True  # ~10% of total
    )

    return X_train, y_train, X_val, y_val, X_test, y_test


[nltk_data] Downloading package stopwords to /home/agata/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/agata/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/agata/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Data

### Small dataset 1K

In [9]:
data_1k = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
data_1k.columns = ['Sentence', 'Class']
data_1k['index'] = data_1k.index                                 
columns = ['index', 'Class', 'Sentence']
data_clean_1k = preprocess_pandas(data_1k, columns)                             
print(f'Example of raw sentence: {data_1k["Sentence"].iloc[0]}')
print(f'Example of cleaned sentence: {data_clean_1k["Sentence"].iloc[0]}')

X_train_1K, y_train_1K, X_val_1K, y_val_1K, X_test_1K, y_test_1K = train_val_test_split(
    data = data_clean_1k['Sentence'].values.astype('U'), 
    labels = data_clean_1k['Class'].values.astype('int32'), 
    train_size=0.80, test_size=0.10, val_size=0.10, 
    seed=RANDOM_SEED)

print(f'Number of training samples: {len(X_train_1K)} and labels: {len(y_train_1K)}')
print(f'Number of validation samples: {len(X_val_1K)} and labels: {len(y_val_1K)}')
print(f'Number of test samples: {len(X_test_1K)} and labels: {len(y_test_1K)}')

Example of raw sentence: so there is no way for me to plug it in here in the us unless i go by a converter
Example of cleaned sentence: way plug us unless go converter
Number of training samples: 810 and labels: 810
Number of validation samples: 90 and labels: 90
Number of test samples: 100 and labels: 100


## Large dataset 25K

In [10]:
data_25k = pd.read_csv("/mnt/12tb_dsk3/agata_dir/adv_dl/lab1/amazon_cells_labelled_LARGE_25K.txt", delimiter='\t', header=None)
data_25k.columns = ['Sentence', 'Class']
data_25k['index'] = data_25k.index                                        
columns = ['index', 'Class', 'Sentence']
data_25k_clean = preprocess_pandas(data_25k, columns)                             

print(f'Example of raw sentence: {data_25k["Sentence"].iloc[0]}')
print(f'Example of cleaned sentence: {data_25k_clean["Sentence"].iloc[0]}')

X_train_25k, y_train_25k, X_val_25k, y_val_25k, X_test_25k, y_test_25k = train_val_test_split(
    data = data_25k_clean['Sentence'].values.astype('U'), 
    labels = data_25k_clean['Class'].values.astype('int32'), 
    train_size=0.80, test_size=0.10, val_size=0.10, 
    seed=RANDOM_SEED)

print(f'Number of training samples: {len(X_train_25k)} and labels: {len(y_train_25k)}')
print(f'Number of validation samples: {len(X_val_25k)} and labels: {len(y_val_25k)}')
print(f'Number of test samples: {len(X_test_25k)} and labels: {len(y_test_25k)}')

Example of raw sentence: ive read this book with much expectation it was very boring all through out the book
Example of cleaned sentence: ive read book much expectation boring book
Number of training samples: 20250 and labels: 20250
Number of validation samples: 2250 and labels: 2250
Number of test samples: 2500 and labels: 2500


## Very large dataset 2GB 

In [11]:
filename = "amazon_reviews_us_Sports_v1_00.tsv"
data_2gb = pd.read_csv(filename,usecols=['review_id', 'star_rating', 'review_body'], delimiter='\t')
print(data_2gb.shape)

# Converst everythig to string
data_2gb['review_id'] = data_2gb['review_id'].astype('str')
data_2gb['star_rating'] = data_2gb['star_rating'].astype('str')
data_2gb['review_body'] = data_2gb['review_body'].astype('str')   

# keep only correct star ratings
star_values = ["1", "1.0", "2", "2.0", "3", "3.0", "4", "4.0", "5", "5.0"]
data_2gb = data_2gb[data_2gb['star_rating'].isin(star_values)]
print(data_2gb.shape)

# convert star rating to numbers 
data_2gb['star_rating'] = data_2gb['star_rating'].astype(float).astype(int)
data_2gb['label'] = data_2gb['star_rating'].apply(lambda x: 1 if x > 3 else 0) # Convert to binary labels
print(data_2gb['label'].value_counts())

data_2gb.rename(columns={'review_body': 'Sentence'}, inplace=True) # rename review_body to Sentence to be consistent with the other datasets
data_2gb.drop(columns=['star_rating'], inplace=True) # drop the original star_rating column as we have the binary label now

data_2gb = data_2gb.dropna() 
data_2gb.shape

/tmp/ipykernel_861442/1675501175.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  data_2gb = pd.read_csv(filename,usecols=['review_id', 'star_rating', 'review_body'], delimiter='\t')


(4833293, 3)
(4833137, 3)
label
1    3863556
0     969581
Name: count, dtype: int64


(4833137, 3)

In [12]:
def preprocess_pandas_v2(data):
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    
    stop_words = set(stopwords.words('english'))
    
    data = data.copy()
    
    data['Sentence'] = (
        data['Sentence']
        .str.lower()
        .str.replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)
        .str.replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\\.|\$)){4}', '', regex=True)
        .str.replace('[^\\w\\s]', '', regex=True)
        .str.replace('\\d', '', regex=True)
    )
    
    def clean_text(text):
        tokens = word_tokenize(text)
        return " ".join([w for w in tokens if w not in stop_words])
    
    data['Sentence'] = data['Sentence'].apply(clean_text)
    
    return data[['review_id', 'label', 'Sentence']]

In [13]:
data_clean_2gb = preprocess_pandas_v2(data_2gb)                             
print(f'Example of raw sentence: {data_2gb["Sentence"].iloc[0]}')
print(f'Example of cleaned sentence: {data_clean_2gb["Sentence"].iloc[0]}')

# Save cleaned data to a new CSV file
data_clean_2gb.to_csv("amazon_reviews_sports_2GB_clean.csv", index=False) 

Example of raw sentence: Bought this last winter I love it! I'm female and the hat is so comfy and fits over my ears perfectly to keep me warm! Love love love this hat! Can't wait for this winter  :)
Example of cleaned sentence: bought last winter love im female hat comfy fits ears perfectly keep warm love love love hat cant wait winter


In [14]:
import numpy as np
from sklearn.model_selection import train_test_split

#data_clean_2gb = pd.read_csv("amazon_reviews_sports_2GB_clean.csv")
#data_clean_2gb.drop(columns=['review_id'], inplace=True)

indices = np.arange(len(data_clean_2gb))

idx_train, idx_test = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED, shuffle=True
)

val_relative_size = 0.2 / (0.8 + 0.2)  # Calculate relative size of validation set with respect to the training set

idx_train, idx_val = train_test_split(
    idx_train, test_size=val_relative_size, random_state=RANDOM_SEED, shuffle=True
)

X_train_2gb = data_clean_2gb['Sentence'].iloc[idx_train]
y_train_2gb = data_clean_2gb['label'].iloc[idx_train]

X_val_2gb = data_clean_2gb['Sentence'].iloc[idx_val]
y_val_2gb = data_clean_2gb['label'].iloc[idx_val]

X_test_2gb = data_clean_2gb['Sentence'].iloc[idx_test]
y_test_2gb = data_clean_2gb['label'].iloc[idx_test]

print(f'Number of training samples: {len(X_train_2gb)} and labels: {len(y_train_2gb)}')
print(f'Number of validation samples: {len(X_val_2gb)} and labels: {len(y_val_2gb)}')
print(f'Number of test samples: {len(X_test_2gb)} and labels: {len(y_test_2gb)}')

Number of training samples: 3093207 and labels: 3093207
Number of validation samples: 773302 and labels: 773302
Number of test samples: 966628 and labels: 966628


# Task 1.1 - ANN and LSTM

## ANN

In [15]:
class SimpleANN(nn.Module):
    def __init__(self, input_size, num_classes, dropout):
        super(SimpleANN, self).__init__()

        self.ann = nn.Sequential(
            nn.Linear(input_size, 128), 
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.ann(x)


### Small dataset

In [16]:
# vectorize data using TFIDF and transform for PyTorch for scalability
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')

training_data_1k = word_vectorizer.fit_transform(X_train_1K).todense()
vocab_size_1k = len(word_vectorizer.vocabulary_)

validation_data_1k = word_vectorizer.transform(X_val_1K).todense()
test_data_1k = word_vectorizer.transform(X_test_1K).todense()

train_x_tensor_1k = torch.from_numpy(np.array(training_data_1k)).type(torch.FloatTensor)
train_y_tensor_1k = torch.from_numpy(np.array(y_train_1K)).long()
validation_x_tensor_1k = torch.from_numpy(np.array(validation_data_1k)).type(torch.FloatTensor)
validation_y_tensor_1k = torch.from_numpy(np.array(y_val_1K)).long()
test_x_tensor_1k = torch.from_numpy(np.array(test_data_1k)).type(torch.FloatTensor)
test_y_tensor_1k = torch.from_numpy(np.array(y_test_1K)).long()

In [17]:
ann_model_small = SimpleANN(input_size=vocab_size_1k, num_classes=2, dropout=0.5).to(device)
print(ann_model_small)

BATCH_SIZE = 32
learning_rate = 1e-3
EPOCHS = 50
train_loader_1k_ann = DataLoader(TensorDataset(train_x_tensor_1k, train_y_tensor_1k), batch_size=BATCH_SIZE, shuffle=True)
val_loader_1k_ann   = DataLoader(TensorDataset(validation_x_tensor_1k, validation_y_tensor_1k),   batch_size=BATCH_SIZE)
test_loader_1k_ann  = DataLoader(TensorDataset(test_x_tensor_1k, test_y_tensor_1k),  batch_size=BATCH_SIZE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model_small.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

wandb.init(
    project="Lab1",
    name="ANN 1K",
    config={
        "learning_rate": learning_rate,
        "architecture": "ANN",
        "dataset": "Amazon Cells 1K",
        "epochs": EPOCHS,
    }
)

# Train
ann_history = train_model(
    ann_model_small,
    train_loader_1k_ann,
    val_loader_1k_ann,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_name="ANN 1K",
    patience=10,
    device=device,
    wandb_logging=True
)

# Evaluate (same run!)
ann_results = evaluate_model(
    ann_model_small,
    test_loader_1k_ann,
    "ANN 1K",
    device,
    wandb_logging=True
)

wandb.finish()

SimpleANN(
  (ann): Sequential(
    (0): Linear(in_features=4678, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)


[ANN 1K] Epoch   1/50 | Train Loss: 0.6978 Acc: 0.5000 | Val Loss: 0.6917 Acc: 0.5111
[ANN 1K] Epoch   2/50 | Train Loss: 0.6810 Acc: 0.5247 | Val Loss: 0.6636 Acc: 0.6333
[ANN 1K] Epoch   3/50 | Train Loss: 0.5936 Acc: 0.8716 | Val Loss: 0.5594 Acc: 0.7667
[ANN 1K] Epoch   4/50 | Train Loss: 0.3353 Acc: 0.9704 | Val Loss: 0.4128 Acc: 0.8222
[ANN 1K] Epoch   5/50 | Train Loss: 0.1071 Acc: 0.9914 | Val Loss: 0.3721 Acc: 0.8333
[ANN 1K] Epoch   6/50 | Train Loss: 0.0407 Acc: 0.9963 | Val Loss: 0.3978 Acc: 0.8111
[ANN 1K] Epoch   7/50 | Train Loss: 0.0186 Acc: 1.0000 | Val Loss: 0.4223 Acc: 0.8222
[ANN 1K] Epoch   8/50 | Train Loss: 0.0104 Acc: 1.0000 | Val Loss: 0.4279 Acc: 0.8222
[ANN 1K] Epoch   9/50 | Train Loss: 0.0077 Acc: 1.0000 | Val Loss: 0.4424 Acc: 0.8222
[ANN 1K] Epoch  10/50 | Train Loss: 0.0055 Acc: 1.0000 | Val Loss: 0.4782 Acc: 0.8222
[ANN 1K] Epoch  11/50 | Train Loss: 0.0044 Acc: 1.0000 | Val Loss: 0.4781 Acc: 0.8222
[ANN 1K] Epoch  12/50 | Train Loss: 0.0039 Acc: 1.0000

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,█████████▁▁▁▁▁▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▁▆████████████
train_loss,██▇▄▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▇████████████
val_loss,█▇▅▂▁▂▂▂▃▃▃▃▃▃▄
epoch,15


### Large dataset

In [18]:
# vectorize data using TFIDF and transform for PyTorch for scalability
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
training_data_25k = word_vectorizer.fit_transform(X_train_25k).todense()
vocab_size_25k = len(word_vectorizer.vocabulary_)
validation_data_25k = word_vectorizer.transform(X_val_25k).todense()
test_data_25k = word_vectorizer.transform(X_test_25k).todense()

train_x_tensor_25k = torch.from_numpy(np.array(training_data_25k)).type(torch.FloatTensor)
train_y_tensor_25k = torch.from_numpy(np.array(y_train_25k)).long()
validation_x_tensor_25k = torch.from_numpy(np.array(validation_data_25k)).type(torch.FloatTensor)
validation_y_tensor_25k = torch.from_numpy(np.array(y_val_25k)).long()
test_x_tensor_25k = torch.from_numpy(np.array(test_data_25k)).type(torch.FloatTensor)
test_y_tensor_25k = torch.from_numpy(np.array(y_test_25k)).long()

In [19]:
ann_model_large = SimpleANN(input_size=vocab_size_25k, num_classes=2,dropout=0.5).to(device)
print(ann_model_large)

BATCH_SIZE = 32
train_loader_25k_ann = DataLoader(TensorDataset(train_x_tensor_25k, train_y_tensor_25k), batch_size=BATCH_SIZE, shuffle=True)
val_loader_25k_ann = DataLoader(TensorDataset(validation_x_tensor_25k, validation_y_tensor_25k), batch_size=BATCH_SIZE)
test_loader_25k_ann = DataLoader(TensorDataset(test_x_tensor_25k, test_y_tensor_25k), batch_size=BATCH_SIZE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ann_model_large.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


wandb.init(
    project="Lab1",
    name="ANN 25K",
    config={
        "learning_rate": learning_rate,
        "architecture": "ANN",
        "dataset": "Amazon Cells 25K",
        "epochs": EPOCHS,
    }
)

# Train
ann_history = train_model(
    ann_model_large,
    train_loader_25k_ann,
    val_loader_25k_ann,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_name="ANN 25k",
    patience=10,
    device=device,
    wandb_logging=True
)

# Evaluate (same run!)
ann_results = evaluate_model(
    ann_model_large,
    test_loader_25k_ann,
    "ANN 25k",
    device,
    wandb_logging=True
)

wandb.finish()

SimpleANN(
  (ann): Sequential(
    (0): Linear(in_features=50000, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)


[ANN 25k] Epoch   1/50 | Train Loss: 0.4095 Acc: 0.8103 | Val Loss: 0.3036 Acc: 0.8751
[ANN 25k] Epoch   2/50 | Train Loss: 0.1866 Acc: 0.9331 | Val Loss: 0.3433 Acc: 0.8636
[ANN 25k] Epoch   3/50 | Train Loss: 0.0925 Acc: 0.9710 | Val Loss: 0.4140 Acc: 0.8613
[ANN 25k] Epoch   4/50 | Train Loss: 0.0486 Acc: 0.9864 | Val Loss: 0.4864 Acc: 0.8591
[ANN 25k] Epoch   5/50 | Train Loss: 0.0341 Acc: 0.9894 | Val Loss: 0.5444 Acc: 0.8564
[ANN 25k] Epoch   6/50 | Train Loss: 0.0344 Acc: 0.9886 | Val Loss: 0.5709 Acc: 0.8573
[ANN 25k] Epoch   7/50 | Train Loss: 0.0391 Acc: 0.9865 | Val Loss: 0.5748 Acc: 0.8542
[ANN 25k] Epoch   8/50 | Train Loss: 0.0392 Acc: 0.9874 | Val Loss: 0.5884 Acc: 0.8564
[ANN 25k] Epoch   9/50 | Train Loss: 0.0373 Acc: 0.9884 | Val Loss: 0.6153 Acc: 0.8547
[ANN 25k] Epoch  10/50 | Train Loss: 0.0308 Acc: 0.9899 | Val Loss: 0.6591 Acc: 0.8564
[ANN 25k] Epoch  11/50 | Train Loss: 0.0244 Acc: 0.9924 | Val Loss: 0.6168 Acc: 0.8564
[ANN 25k] Early stopping at epoch 11

=====

epoch,▁▂▂▃▄▅▅▆▇▇█
lr,█████████▁▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▆▇████████
train_loss,█▄▂▁▁▁▁▁▁▁▁
val_acc,█▄▃▃▂▂▁▂▁▂▂
val_loss,▁▂▃▅▆▆▆▇▇█▇
epoch,11


### Very large dataset

In [20]:
# # vectorize data using TFIDF and transform for PyTorch for scalability
# word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=10000, max_df=0.5, use_idf=True, norm='l2')

# # training_data_2gb = word_vectorizer.fit_transform(X_train_2gb)
# vocab_size_2gb = len(word_vectorizer.vocabulary_)
# # print(vocab_size_2gb)

# # validation_data_2gb = word_vectorizer.transform(X_val_2gb)
# # test_data_2gb = word_vectorizer.transform(X_test_2gb)
# # train_x_tensor_2gb = torch.from_numpy(np.array(training_data_2gb)).type(torch.FloatTensor)
# # train_y_tensor_2gb = torch.from_numpy(np.array(y_train_2gb)).long()
# # validation_x_tensor_2gb = torch.from_numpy(np.array(validation_data_2gb)).type(torch.FloatTensor)
# # validation_y_tensor_2gb = torch.from_numpy(np.array(y_val_2gb)).long()
# # test_x_tensor_2gb = torch.from_numpy(np.array(test_data_2gb)).type(torch.FloatTensor)
# # test_y_tensor_2gb = torch.from_numpy(np.array(y_test_2gb)).long()

# training_data_2gb = word_vectorizer.fit_transform(X_train_2gb).toarray().astype(np.float32)
# validation_data_2gb = word_vectorizer.transform(X_val_2gb).toarray().astype(np.float32)
# test_data_2gb = word_vectorizer.transform(X_test_2gb).toarray().astype(np.float32)

# train_x_tensor_2gb = torch.from_numpy(training_data_2gb)
# validation_x_tensor_2gb = torch.from_numpy(validation_data_2gb)
# test_x_tensor_2gb = torch.from_numpy(test_data_2gb)

# train_y_tensor_2gb = torch.tensor(y_train_2gb.to_numpy(), dtype=torch.long)
# validation_y_tensor_2gb = torch.tensor(y_val_2gb.to_numpy(), dtype=torch.long)
# test_y_tensor_2gb = torch.tensor(y_test_2gb.to_numpy(), dtype=torch.long)

In [21]:
# ann_model_very_large = SimpleANN(input_size=vocab_size_2gb, num_classes=2,dropout=0.5).to(device)
# print(ann_model_very_large)

# EPOCHS = 30
# BATCH_SIZE = 32
# train_loader_2gb_ann = DataLoader(TensorDataset(train_x_tensor_2gb, train_y_tensor_2gb), batch_size=BATCH_SIZE, shuffle=True)
# val_loader_2gb_ann = DataLoader(TensorDataset(validation_x_tensor_2gb, validation_y_tensor_2gb), batch_size=BATCH_SIZE)
# test_loader_2gb_ann = DataLoader(TensorDataset(test_x_tensor_2gb, test_y_tensor_2gb), batch_size=BATCH_SIZE)

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(ann_model_very_large.parameters(), lr=1e-3, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


# wandb.init(
#     project="Lab1",
#     name="ANN 2GB",
#     config={
#         "learning_rate": learning_rate,
#         "architecture": "ANN",
#         "dataset": "Amazon Cells 2GB",
#         "epochs": EPOCHS,
#     }
# )

# # Train
# ann_history = train_model(
#     ann_model_very_large,
#     train_loader_2gb_ann,
#     val_loader_2gb_ann,
#     epochs=EPOCHS,
#     criterion=criterion,
#     optimizer=optimizer,
#     scheduler=scheduler,
#     model_name="ANN 2GB",
#     patience=10,
#     device=device,
#     wandb_logging=True
# )

# # Evaluate (same run!)
# ann_results = evaluate_model(
#     ann_model_very_large,
#     test_loader_2gb_ann,
#     "ANN 2GB",
#     device,
#     wandb_logging=True
# )

# wandb.finish()

## LSTM

In [22]:
class BiLSTM(nn.Module):
    def __init__(self, vocabulary, hidden_size, num_classes, dropout, lstm_layers, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=len(vocabulary), embedding_dim=embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size=hidden_size, num_layers=lstm_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size*2, num_classes) 
    
    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x) 
        x, _ = torch.max(x, dim=1) # Global max pooling over time steps
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [23]:
from collections import Counter

PAD_IDX = 0
UNK_IDX = 1

def build_vocab(texts, min_freq):
    counter = Counter()
    for sentence in texts:
        counter.update(sentence.split())
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for w, c in counter.items():
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len):
    ids = [vocab.get(tok, UNK_IDX) for tok in text.split()][:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

### Small dataset

In [24]:
vocab_1k = build_vocab(X_train_1K, min_freq=2) # keep only words that appear at least 2 times in the training set
max_len = int(np.percentile([len(sentence.split()) for sentence in X_train_1K], 95))  # most sentences are kept full, but very long ones are truncated. Better memory efficiency and lower training time
print(f"Max sequence length: {max_len}")
print(f"Vocabulary size: {len(vocab_1k)}")

X_train_tensor_1k = torch.tensor([encode_text(s, vocab_1k, max_len) for s in X_train_1K], dtype=torch.long)
X_val_tensor_1k   = torch.tensor([encode_text(s, vocab_1k, max_len) for s in X_val_1K], dtype=torch.long)
X_test_tensor_1k  = torch.tensor([encode_text(s, vocab_1k, max_len) for s in X_test_1K], dtype=torch.long)

y_train_tensor_1k = torch.tensor(y_train_1K, dtype=torch.long)
y_val_tensor_1k   = torch.tensor(y_val_1K, dtype=torch.long)
y_test_tensor_1k  = torch.tensor(y_test_1K, dtype=torch.long)


print(f"Training data size: {len(X_train_tensor_1k)}, Length: {X_train_tensor_1k.shape[1]}")
print(f"Validation data size: {len(X_val_tensor_1k)}, Length: {X_val_tensor_1k.shape[1]}")
print(f"Test data size: {len(X_test_tensor_1k)}, Length: {X_test_tensor_1k.shape[1]}")

BATCH_SIZE = 32
train_loader_1k_lstm = DataLoader(TensorDataset(X_train_tensor_1k, y_train_tensor_1k), batch_size=BATCH_SIZE, shuffle=True)
val_loader_1k_lstm   = DataLoader(TensorDataset(X_val_tensor_1k, y_val_tensor_1k), batch_size=BATCH_SIZE)
test_loader_1k_lstm  = DataLoader(TensorDataset(X_test_tensor_1k, y_test_tensor_1k), batch_size=BATCH_SIZE)

Max sequence length: 12
Vocabulary size: 574
Training data size: 810, Length: 12
Validation data size: 90, Length: 12
Test data size: 100, Length: 12


In [25]:
lstm_model_small = BiLSTM(vocabulary=vocab_1k, hidden_size=32, num_classes=2, dropout=0.5, lstm_layers=1, embed_dim=32).to(device)
lstm_model_small

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model_small.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

wandb.init(
    project="Lab1",
    name="LSTM 1K",
    config={
        "learning_rate": learning_rate,
        "architecture": "LSTM",
        "dataset": "Amazon Cells 1K",
        "epochs": EPOCHS,
    }
)

# Train
lstm_history = train_model(
    lstm_model_small,
    train_loader_1k_lstm,
    val_loader_1k_lstm,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_name="LSTM 1K",
    patience=10,
    device=device,
    wandb_logging=True
)

# Evaluate (same run!)
lstm_test_results = evaluate_model(
    lstm_model_small,
    test_loader_1k_lstm,
    "LSTM 1K",
    device,
    wandb_logging=True
)

wandb.finish()

/home/agata/miniconda3/envs/nnlm-gpu/lib/python3.10/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


[LSTM 1K] Epoch   1/50 | Train Loss: 0.6934 Acc: 0.5284 | Val Loss: 0.6846 Acc: 0.5778
[LSTM 1K] Epoch   2/50 | Train Loss: 0.6779 Acc: 0.5901 | Val Loss: 0.6750 Acc: 0.6222
[LSTM 1K] Epoch   3/50 | Train Loss: 0.6609 Acc: 0.6222 | Val Loss: 0.6661 Acc: 0.6222
[LSTM 1K] Epoch   4/50 | Train Loss: 0.6421 Acc: 0.6667 | Val Loss: 0.6525 Acc: 0.6333
[LSTM 1K] Epoch   5/50 | Train Loss: 0.6289 Acc: 0.6716 | Val Loss: 0.6380 Acc: 0.6333
[LSTM 1K] Epoch   6/50 | Train Loss: 0.6005 Acc: 0.7012 | Val Loss: 0.6261 Acc: 0.6333
[LSTM 1K] Epoch   7/50 | Train Loss: 0.5802 Acc: 0.7235 | Val Loss: 0.6063 Acc: 0.6556
[LSTM 1K] Epoch   8/50 | Train Loss: 0.5403 Acc: 0.7494 | Val Loss: 0.5946 Acc: 0.6444
[LSTM 1K] Epoch   9/50 | Train Loss: 0.5032 Acc: 0.7580 | Val Loss: 0.5859 Acc: 0.7111
[LSTM 1K] Epoch  10/50 | Train Loss: 0.4716 Acc: 0.7901 | Val Loss: 0.5537 Acc: 0.7111
[LSTM 1K] Epoch  11/50 | Train Loss: 0.4398 Acc: 0.8173 | Val Loss: 0.5486 Acc: 0.7444
[LSTM 1K] Epoch  12/50 | Train Loss: 0.4133

epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
lr,█████████▃▃▃▃▃▃▃▃▃▃▁▁▁▁▁▁▁▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇▇▇▇█▇██████
train_loss,███▇▇▇▆▆▅▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▃▃▃▃▄▃▆▆▇▇▆▆▇▇▇▇██████▇█▇
val_loss,██▇▇▆▆▅▄▄▂▂▂▂▂▂▂▁▁▂▂▂▂▃▃▃▄▄
epoch,27


### Large dataset

In [26]:
vocab_large = build_vocab(X_train_25k, min_freq=2)
max_len_large = int(np.percentile([len(sentence.split()) for sentence in X_train_25k], 95))
print(f"Max sequence length: {max_len_large}")
print(f"Vocabulary size: {len(vocab_large)}")

X_train_tensor_25k = torch.tensor([encode_text(s, vocab_large, max_len_large) for s in X_train_25k], dtype=torch.long)
X_val_tensor_25k   = torch.tensor([encode_text(s, vocab_large, max_len_large) for s in X_val_25k], dtype=torch.long)
X_test_tensor_25k  = torch.tensor([encode_text(s, vocab_large, max_len_large) for s in X_test_25k], dtype=torch.long)

y_train_tensor_25k = torch.tensor(y_train_25k, dtype=torch.long)
y_val_tensor_25k   = torch.tensor(y_val_25k, dtype=torch.long)
y_test_tensor_25k  = torch.tensor(y_test_25k, dtype=torch.long)

print(f"Training data size: {len(X_train_tensor_25k)}, Length: {X_train_tensor_25k.shape[1]}")
print(f"Validation data size: {len(X_val_tensor_25k)}, Length: {X_val_tensor_25k.shape[1]}")
print(f"Test data size: {len(X_test_tensor_25k)}, Length: {X_test_tensor_25k.shape[1]}")

BATCH_SIZE = 32
train_loader_25k_lstm = DataLoader(TensorDataset(X_train_tensor_25k, y_train_tensor_25k), batch_size=BATCH_SIZE, shuffle=True)
val_loader_25k_lstm   = DataLoader(TensorDataset(X_val_tensor_25k, y_val_tensor_25k), batch_size=BATCH_SIZE)
test_loader_25k_lstm  = DataLoader(TensorDataset(X_test_tensor_25k, y_test_tensor_25k), batch_size=BATCH_SIZE)

Max sequence length: 13
Vocabulary size: 8954
Training data size: 20250, Length: 13
Validation data size: 2250, Length: 13
Test data size: 2500, Length: 13


In [27]:
lstm_model_large = BiLSTM(vocabulary=vocab_large, hidden_size=32, num_classes=2, dropout=0.5, lstm_layers=1, embed_dim=32).to(device)
lstm_model_large

criterion = nn.CrossEntropyLoss()
learning_rate = 1e-3
optimizer = optim.Adam(lstm_model_large.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


wandb.init(
    project="Lab1",
    name="LSTM 25K",
    config={
        "learning_rate": learning_rate,
        "architecture": "LSTM",
        "dataset": "Amazon Cells 25K",
        "epochs": EPOCHS,
    }
)

# Train
lstm_history = train_model(
    lstm_model_large,
    train_loader_25k_lstm,
    val_loader_25k_lstm,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_name="LSTM 25K",
    patience=10,
    device=device,
    wandb_logging=True
)

# Evaluate (same run!)
lstm_test_results = evaluate_model(
    lstm_model_large,
    test_loader_25k_lstm,
    "LSTM 25K",
    device,
    wandb_logging=True
)

wandb.finish()

/home/agata/miniconda3/envs/nnlm-gpu/lib/python3.10/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


[LSTM 25K] Epoch   1/50 | Train Loss: 0.5617 Acc: 0.6933 | Val Loss: 0.4394 Acc: 0.8049
[LSTM 25K] Epoch   2/50 | Train Loss: 0.4089 Acc: 0.8146 | Val Loss: 0.3685 Acc: 0.8453
[LSTM 25K] Epoch   3/50 | Train Loss: 0.3389 Acc: 0.8537 | Val Loss: 0.3336 Acc: 0.8582
[LSTM 25K] Epoch   4/50 | Train Loss: 0.2920 Acc: 0.8796 | Val Loss: 0.3356 Acc: 0.8609
[LSTM 25K] Epoch   5/50 | Train Loss: 0.2612 Acc: 0.8972 | Val Loss: 0.3313 Acc: 0.8644
[LSTM 25K] Epoch   6/50 | Train Loss: 0.2354 Acc: 0.9115 | Val Loss: 0.3488 Acc: 0.8631
[LSTM 25K] Epoch   7/50 | Train Loss: 0.2161 Acc: 0.9207 | Val Loss: 0.3499 Acc: 0.8560
[LSTM 25K] Epoch   8/50 | Train Loss: 0.1965 Acc: 0.9297 | Val Loss: 0.4059 Acc: 0.8507
[LSTM 25K] Epoch   9/50 | Train Loss: 0.1864 Acc: 0.9327 | Val Loss: 0.3875 Acc: 0.8396
[LSTM 25K] Epoch  10/50 | Train Loss: 0.1695 Acc: 0.9402 | Val Loss: 0.4489 Acc: 0.8427
[LSTM 25K] Epoch  11/50 | Train Loss: 0.1378 Acc: 0.9535 | Val Loss: 0.4776 Acc: 0.8480
[LSTM 25K] Epoch  12/50 | Train 

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,█████████▁▁▁▁▁▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▄▅▆▆▇▇▇▇▇█████
train_loss,█▆▅▄▃▃▃▂▂▂▂▁▁▁▁
val_acc,▁▆▇███▇▆▅▅▆▅▅▅▅
val_loss,▄▂▁▁▁▁▁▃▃▄▅▅▆▇█
epoch,15


### Very large dataset

In [28]:
vocab_very_large = build_vocab(X_train_2gb, min_freq=10)
max_len_very_large = int(np.percentile([len(sentence.split()) for sentence in X_train_2gb[1]], 95))
print(f"Max sequence length: {max_len_very_large}")
print(f"Vocabulary size: {len(vocab_very_large)}")

X_train_tensor_2gb = torch.tensor([encode_text(s, vocab_very_large, max_len_very_large) for s in X_train_2gb], dtype=torch.long)
X_val_tensor_2gb   = torch.tensor([encode_text(s, vocab_very_large, max_len_very_large) for s in X_val_2gb], dtype=torch.long)
X_test_tensor_2gb  = torch.tensor([encode_text(s, vocab_very_large, max_len_very_large) for s in X_test_2gb], dtype=torch.long)

y_train_tensor_2gb = torch.tensor(y_train_2gb.to_numpy(), dtype=torch.long)
y_val_tensor_2gb   = torch.tensor(y_val_2gb.to_numpy(), dtype=torch.long)
y_test_tensor_2gb  = torch.tensor(y_test_2gb.to_numpy(), dtype=torch.long)

print(f"Training data size: {len(X_train_tensor_2gb)}, Length: {X_train_tensor_2gb.shape[1]}")
print(f"Validation data size: {len(X_val_tensor_2gb)}, Length: {X_val_tensor_2gb.shape[1]}")
print(f"Test data size: {len(X_test_tensor_2gb)}, Length: {X_test_tensor_2gb.shape[1]}")



Max sequence length: 1
Vocabulary size: 63603
Training data size: 3093207, Length: 1
Validation data size: 773302, Length: 1
Test data size: 966628, Length: 1


In [29]:
BATCH_SIZE = 32
EPOCHS = 30
train_loader_2gb_lstm = DataLoader(TensorDataset(X_train_tensor_2gb, y_train_tensor_2gb), batch_size=BATCH_SIZE, shuffle=True)
val_loader_2gb_lstm   = DataLoader(TensorDataset(X_val_tensor_2gb, y_val_tensor_2gb), batch_size=BATCH_SIZE)
test_loader_2gb_lstm  = DataLoader(TensorDataset(X_test_tensor_2gb, y_test_tensor_2gb), batch_size=BATCH_SIZE)

lstm_model_very_large = BiLSTM(vocabulary=vocab_very_large, hidden_size=32, num_classes=2, dropout=0.5, lstm_layers=1, embed_dim=32).to(device)
print(lstm_model_very_large)

learning_rate = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model_very_large.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

wandb.init(
    project="Lab1",
    name="LSTM 2GB",
    config={
        "learning_rate": learning_rate,
        "architecture": "LSTM",
        "dataset": "Amazon Cells 2GB",
        "epochs": EPOCHS,
        
    },
)

# Train
lstm_history = train_model(
    lstm_model_very_large,
    train_loader_2gb_lstm,
    val_loader_2gb_lstm,
    epochs=EPOCHS,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_name="LSTM 2GB",
    patience=10,
    device=device,
    wandb_logging=True
)

# Evaluate (same run!)
lstm_test_results = evaluate_model(
    lstm_model_very_large,
    test_loader_2gb_lstm,
    "LSTM 2GB",
    device,
    wandb_logging=True
)

wandb.finish()

BiLSTM(
  (embedding): Embedding(63603, 32, padding_idx=0)
  (lstm): LSTM(32, 32, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


/home/agata/miniconda3/envs/nnlm-gpu/lib/python3.10/site-packages/torch/nn/modules/rnn.py:88: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


[LSTM 2GB] Epoch   1/30 | Train Loss: 0.4271 Acc: 0.8155 | Val Loss: 0.4216 Acc: 0.8170
[LSTM 2GB] Epoch   2/30 | Train Loss: 0.4233 Acc: 0.8170 | Val Loss: 0.4217 Acc: 0.8174
[LSTM 2GB] Epoch   3/30 | Train Loss: 0.4230 Acc: 0.8172 | Val Loss: 0.4217 Acc: 0.8178
[LSTM 2GB] Epoch   4/30 | Train Loss: 0.4230 Acc: 0.8172 | Val Loss: 0.4211 Acc: 0.8177
[LSTM 2GB] Epoch   5/30 | Train Loss: 0.4229 Acc: 0.8172 | Val Loss: 0.4212 Acc: 0.8175
[LSTM 2GB] Epoch   6/30 | Train Loss: 0.4230 Acc: 0.8173 | Val Loss: 0.4213 Acc: 0.8175
[LSTM 2GB] Epoch   7/30 | Train Loss: 0.4230 Acc: 0.8173 | Val Loss: 0.4212 Acc: 0.8174
[LSTM 2GB] Epoch   8/30 | Train Loss: 0.4230 Acc: 0.8173 | Val Loss: 0.4213 Acc: 0.8177
[LSTM 2GB] Epoch   9/30 | Train Loss: 0.4229 Acc: 0.8173 | Val Loss: 0.4211 Acc: 0.8174
[LSTM 2GB] Epoch  10/30 | Train Loss: 0.4229 Acc: 0.8173 | Val Loss: 0.4215 Acc: 0.8173
[LSTM 2GB] Epoch  11/30 | Train Loss: 0.4218 Acc: 0.8176 | Val Loss: 0.4210 Acc: 0.8176
[LSTM 2GB] Epoch  12/30 | Train 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████████▄▄▄▄▄▄▄▄▄▄▂▂▂▂▂▂▂▂▂▂▁
test_accuracy,▁
test_f1,▁
test_precision,▁
test_recall,▁
train_acc,▁▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██▇▇██▇██
train_loss,█▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▆▅▄▄▃▅▃▃▅▆▆█▇▆▅█▇▆▇█████▇██▇
val_loss,███▆▆▆▆▆▆▇▆▇▃▃▃▄▃▃▃▃▂▁▂▂▁▁▁▂▂▁
epoch,30


# Task 1.2 - Transformer implementation 

In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

distilbert = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(distilbert)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    distilbert,
    num_labels=2
)

roberta = "roberta-base" # bert did not work: OSError: bert is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
tokenizer_roberta = AutoTokenizer.from_pretrained(roberta)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta,
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./tramnsformers_output",
    eval_strategy="epoch",
    learning_rate=2e-5, # small learning rate for fine-tuning
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_steps=100, # learning rate warmup for stability in early training
    disable_tqdm=True, # to avoid RuntimeError: on_train_begin must be called before on_evaluate
    # use wandb for logging (optional, but useful for tracking training progress and metrics
    report_to="wandb",


)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

def tokenize_bert(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

def tokenize_roberta(texts):
    return tokenizer_roberta(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Small dataset

In [31]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_train_1K,
    "label": y_train_1K
})

val_dataset = Dataset.from_dict({
    "text": X_val_1K,
    "label": y_val_1K
})

test_dataset = Dataset.from_dict({
    "text": X_test_1K,
    "label": y_test_1K
})

train_dataset_1k_bert = train_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
val_dataset_1k_bert   = val_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
test_dataset_1k_bert = test_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)

train_dataset_1k_bert.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_1k_bert.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_1k_bert.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset_1k_roberta = train_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
val_dataset_1k_roberta   = val_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
test_dataset_1k_roberta = test_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)

train_dataset_1k_roberta.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_1k_roberta.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_1k_roberta.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/810 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [32]:
trainer_bert_1k = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset_1k_bert,
    eval_dataset=val_dataset_1k_bert,
    compute_metrics=compute_metrics
)

trainer_bert_1k.train()   
trainer_bert_1k.evaluate(test_dataset_1k_bert)

{'eval_loss': '0.6913', 'eval_accuracy': '0.5333', 'eval_f1': '0.4615', 'eval_precision': '0.5625', 'eval_recall': '0.3913', 'eval_runtime': '0.0479', 'eval_samples_per_second': '1879', 'eval_steps_per_second': '62.62', 'epoch': '1'}
{'eval_loss': '0.655', 'eval_accuracy': '0.7333', 'eval_f1': '0.7', 'eval_precision': '0.8235', 'eval_recall': '0.6087', 'eval_runtime': '0.0469', 'eval_samples_per_second': '1919', 'eval_steps_per_second': '63.97', 'epoch': '2'}
{'eval_loss': '0.4071', 'eval_accuracy': '0.8222', 'eval_f1': '0.84', 'eval_precision': '0.7778', 'eval_recall': '0.913', 'eval_runtime': '0.047', 'eval_samples_per_second': '1915', 'eval_steps_per_second': '63.82', 'epoch': '3'}
{'eval_loss': '0.3614', 'eval_accuracy': '0.8556', 'eval_f1': '0.8602', 'eval_precision': '0.8511', 'eval_recall': '0.8696', 'eval_runtime': '0.0473', 'eval_samples_per_second': '1904', 'eval_steps_per_second': '63.46', 'epoch': '4'}
{'eval_loss': '0.3696', 'eval_accuracy': '0.8667', 'eval_f1': '0.8723', 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.4978', 'eval_accuracy': '0.8667', 'eval_f1': '0.8696', 'eval_precision': '0.8696', 'eval_recall': '0.8696', 'eval_runtime': '0.0448', 'eval_samples_per_second': '2007', 'eval_steps_per_second': '66.91', 'epoch': '10'}
{'train_runtime': '20.42', 'train_samples_per_second': '396.6', 'train_steps_per_second': '12.73', 'train_loss': '0.2684', 'epoch': '10'}
{'eval_loss': '0.786', 'eval_accuracy': '0.81', 'eval_f1': '0.8155', 'eval_precision': '0.7778', 'eval_recall': '0.8571', 'eval_runtime': '0.056', 'eval_samples_per_second': '1786', 'eval_steps_per_second': '71.42', 'epoch': '10'}


{'eval_loss': 0.7859556078910828,
 'eval_accuracy': 0.81,
 'eval_f1': 0.8155339805825242,
 'eval_precision': 0.7777777777777778,
 'eval_recall': 0.8571428571428571,
 'eval_runtime': 0.056,
 'eval_samples_per_second': 1785.608,
 'eval_steps_per_second': 71.424,
 'epoch': 10.0}

In [33]:
trainer_roberta_1k = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset_1k_roberta,
    eval_dataset=val_dataset_1k_roberta,
    compute_metrics=compute_metrics
)

trainer_roberta_1k.train()                 
trainer_roberta_1k.evaluate(test_dataset_1k_roberta)   # test

{'eval_loss': '0.6972', 'eval_accuracy': '0.5111', 'eval_f1': '0.6765', 'eval_precision': '0.5111', 'eval_recall': '1', 'eval_runtime': '0.0892', 'eval_samples_per_second': '1009', 'eval_steps_per_second': '33.62', 'epoch': '1'}
{'eval_loss': '0.6751', 'eval_accuracy': '0.6667', 'eval_f1': '0.7321', 'eval_precision': '0.6212', 'eval_recall': '0.8913', 'eval_runtime': '0.0893', 'eval_samples_per_second': '1008', 'eval_steps_per_second': '33.6', 'epoch': '2'}
{'eval_loss': '0.3796', 'eval_accuracy': '0.8556', 'eval_f1': '0.8539', 'eval_precision': '0.8837', 'eval_recall': '0.8261', 'eval_runtime': '0.0918', 'eval_samples_per_second': '980.8', 'eval_steps_per_second': '32.69', 'epoch': '3'}
{'eval_loss': '0.4723', 'eval_accuracy': '0.8667', 'eval_f1': '0.8846', 'eval_precision': '0.7931', 'eval_recall': '1', 'eval_runtime': '0.0902', 'eval_samples_per_second': '997.8', 'eval_steps_per_second': '33.26', 'epoch': '4'}
{'eval_loss': '0.3083', 'eval_accuracy': '0.8889', 'eval_f1': '0.8958', '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.4647', 'eval_accuracy': '0.8889', 'eval_f1': '0.8936', 'eval_precision': '0.875', 'eval_recall': '0.913', 'eval_runtime': '0.0811', 'eval_samples_per_second': '1110', 'eval_steps_per_second': '36.99', 'epoch': '10'}
{'train_runtime': '37.58', 'train_samples_per_second': '215.5', 'train_steps_per_second': '6.919', 'train_loss': '0.308', 'epoch': '10'}
{'eval_loss': '0.595', 'eval_accuracy': '0.89', 'eval_f1': '0.8911', 'eval_precision': '0.8654', 'eval_recall': '0.9184', 'eval_runtime': '0.1007', 'eval_samples_per_second': '993.2', 'eval_steps_per_second': '39.73', 'epoch': '10'}


{'eval_loss': 0.5949832797050476,
 'eval_accuracy': 0.89,
 'eval_f1': 0.8910891089108911,
 'eval_precision': 0.8653846153846154,
 'eval_recall': 0.9183673469387755,
 'eval_runtime': 0.1007,
 'eval_samples_per_second': 993.16,
 'eval_steps_per_second': 39.726,
 'epoch': 10.0}

### Large dataset

In [34]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_train_25k,
    "label": y_train_25k
})

val_dataset = Dataset.from_dict({
    "text": X_val_25k,
    "label": y_val_25k
})

# evaluate on test set
test_dataset = Dataset.from_dict({
    "text": X_test_25k,
    "label": y_test_25k
})

train_dataset_bert_25k = train_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
val_dataset_bert_25k   = val_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
test_dataset_bert_25k = test_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
train_dataset_bert_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_bert_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_bert_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset_roberta_25k = train_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
val_dataset_roberta_25k   = val_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
test_dataset_roberta_25k = test_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)    
train_dataset_roberta_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_roberta_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_roberta_25k.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/20250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/20250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [35]:
trainer_bert_25k = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset_bert_25k,
    eval_dataset=val_dataset_bert_25k,
    compute_metrics=compute_metrics
)

trainer_bert_25k.train()                 # initializes everything
trainer_bert_25k.evaluate(test_dataset_bert_25k)   # test

{'loss': '0.3825', 'grad_norm': '5.162', 'learning_rate': '1.872e-05', 'epoch': '0.7899'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3', 'eval_accuracy': '0.8742', 'eval_f1': '0.8907', 'eval_precision': '0.9202', 'eval_recall': '0.863', 'eval_runtime': '1.304', 'eval_samples_per_second': '1726', 'eval_steps_per_second': '54.46', 'epoch': '1'}
{'loss': '0.268', 'grad_norm': '6.114', 'learning_rate': '1.711e-05', 'epoch': '1.58'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3228', 'eval_accuracy': '0.8684', 'eval_f1': '0.8933', 'eval_precision': '0.8616', 'eval_recall': '0.9274', 'eval_runtime': '1.324', 'eval_samples_per_second': '1700', 'eval_steps_per_second': '53.64', 'epoch': '2'}
{'loss': '0.2117', 'grad_norm': '9.974', 'learning_rate': '1.551e-05', 'epoch': '2.37'}


...

{'eval_loss': '0.8577', 'eval_accuracy': '0.8742', 'eval_f1': '0.8954', 'eval_precision': '0.8846', 'eval_recall': '0.9064', 'eval_runtime': '1.32', 'eval_samples_per_second': '1705', 'eval_steps_per_second': '53.8', 'epoch': '9'}
{'loss': '0.01675', 'grad_norm': '0.08942', 'learning_rate': '1.063e-06', 'epoch': '9.479'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.8865', 'eval_accuracy': '0.876', 'eval_f1': '0.8966', 'eval_precision': '0.8877', 'eval_recall': '0.9057', 'eval_runtime': '1.288', 'eval_samples_per_second': '1747', 'eval_steps_per_second': '55.13', 'epoch': '10'}
{'train_runtime': '416.6', 'train_samples_per_second': '486', 'train_steps_per_second': '15.19', 'train_loss': '0.1093', 'epoch': '10'}
{'eval_loss': '1.002', 'eval_accuracy': '0.8612', 'eval_f1': '0.887', 'eval_precision': '0.8725', 'eval_recall': '0.902', 'eval_runtime': '1.45', 'eval_samples_per_second': '1725', 'eval_steps_per_second': '54.5', 'epoch': '10'}


{'eval_loss': 1.001971960067749,
 'eval_accuracy': 0.8612,
 'eval_f1': 0.8870074894171279,
 'eval_precision': 0.8725176169122357,
 'eval_recall': 0.9019867549668874,
 'eval_runtime': 1.4496,
 'eval_samples_per_second': 1724.583,
 'eval_steps_per_second': 54.497,
 'epoch': 10.0}

In [36]:
trainer_roberta_25k = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset_roberta_25k,
    eval_dataset=val_dataset_roberta_25k,
    compute_metrics=compute_metrics
)

trainer_roberta_25k.train()
trainer_roberta_25k.evaluate(test_dataset_roberta_25k)  # test

{'loss': '0.3618', 'grad_norm': '14.8', 'learning_rate': '1.872e-05', 'epoch': '0.7899'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.27', 'eval_accuracy': '0.8884', 'eval_f1': '0.9053', 'eval_precision': '0.9125', 'eval_recall': '0.8982', 'eval_runtime': '2.309', 'eval_samples_per_second': '974.5', 'eval_steps_per_second': '30.75', 'epoch': '1'}
{'loss': '0.2754', 'grad_norm': '10.1', 'learning_rate': '1.711e-05', 'epoch': '1.58'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.3034', 'eval_accuracy': '0.8862', 'eval_f1': '0.909', 'eval_precision': '0.8659', 'eval_recall': '0.9566', 'eval_runtime': '2.329', 'eval_samples_per_second': '966', 'eval_steps_per_second': '30.48', 'epoch': '2'}
{'loss': '0.2204', 'grad_norm': '8.544', 'learning_rate': '1.551e-05', 'epoch': '2.37'}


...

{'eval_loss': '0.6192', 'eval_accuracy': '0.896', 'eval_f1': '0.9155', 'eval_precision': '0.8842', 'eval_recall': '0.9491', 'eval_runtime': '2.319', 'eval_samples_per_second': '970.1', 'eval_steps_per_second': '30.61', 'epoch': '9'}
{'loss': '0.0464', 'grad_norm': '0.1091', 'learning_rate': '1.063e-06', 'epoch': '9.479'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.6373', 'eval_accuracy': '0.8951', 'eval_f1': '0.9135', 'eval_precision': '0.8951', 'eval_recall': '0.9326', 'eval_runtime': '2.255', 'eval_samples_per_second': '997.6', 'eval_steps_per_second': '31.48', 'epoch': '10'}
{'train_runtime': '836.8', 'train_samples_per_second': '242', 'train_steps_per_second': '7.564', 'train_loss': '0.1386', 'epoch': '10'}
{'eval_loss': '0.6521', 'eval_accuracy': '0.8908', 'eval_f1': '0.9119', 'eval_precision': '0.8897', 'eval_recall': '0.9351', 'eval_runtime': '2.53', 'eval_samples_per_second': '988.3', 'eval_steps_per_second': '31.23', 'epoch': '10'}


{'eval_loss': 0.6521254181861877,
 'eval_accuracy': 0.8908,
 'eval_f1': 0.9118501775912173,
 'eval_precision': 0.8897290485192186,
 'eval_recall': 0.9350993377483444,
 'eval_runtime': 2.5295,
 'eval_samples_per_second': 988.334,
 'eval_steps_per_second': 31.231,
 'epoch': 10.0}

## Very large dataset

In [37]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_train_2gb,
    "label": y_train_2gb
})

val_dataset = Dataset.from_dict({
    "text": X_val_2gb,
    "label": y_val_2gb
})

# evaluate on test set
test_dataset = Dataset.from_dict({
    "text": X_test_2gb,
    "label": y_test_2gb
})

train_dataset_bert_2gb = train_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
val_dataset_bert_2gb   = val_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
test_dataset_bert_2gb = test_dataset.map(lambda x: tokenize_bert(x["text"]), batched=True)
train_dataset_bert_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_bert_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_bert_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_dataset_roberta_2gb = train_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
val_dataset_roberta_2gb   = val_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)
test_dataset_roberta_2gb = test_dataset.map(lambda x: tokenize_roberta(x["text"]), batched=True)    
train_dataset_roberta_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_roberta_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset_roberta_2gb.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/3093207 [00:00<?, ? examples/s]

Map:   0%|          | 0/773302 [00:00<?, ? examples/s]

Map:   0%|          | 0/966628 [00:00<?, ? examples/s]

Map:   0%|          | 0/3093207 [00:00<?, ? examples/s]

Map:   0%|          | 0/773302 [00:00<?, ? examples/s]

Map:   0%|          | 0/966628 [00:00<?, ? examples/s]

In [38]:
trainer_bert_2gb = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset_bert_2gb,
    eval_dataset=val_dataset_bert_2gb,
    compute_metrics=compute_metrics
)

trainer_bert_2gb.train()
trainer_bert_2gb.evaluate(test_dataset_bert_2gb)

{'loss': '0.3704', 'grad_norm': '3.74', 'learning_rate': '1.999e-05', 'epoch': '0.005173'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2698', 'grad_norm': '3.104', 'learning_rate': '1.998e-05', 'epoch': '0.01035'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2618', 'grad_norm': '2.348', 'learning_rate': '1.997e-05', 'epoch': '0.01552'}


...

{'loss': '0.1037', 'grad_norm': '6.371', 'learning_rate': '9.873e-06', 'epoch': '5.064'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7807bcffb3d0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7807be3ef6a0, execution_count=38 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7807be3efeb0, raw_cell="trainer_bert_2gb = Trainer(
    model=distilbert_m.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B129.16.110.61/mnt/12tb_dsk3/agata_dir/adv_dl/lab1/lab1_agata.ipynb#Y103sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
trainer_roberta_2gb = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset_roberta_2gb,
    eval_dataset=val_dataset_roberta_2gb,
    compute_metrics=compute_metrics
)

trainer_roberta_2gb.train()
trainer_roberta_2gb.evaluate(test_dataset_roberta_2gb)

# Task 1.3 - Comparison

 'eval_accuracy': 0.8768,
 'eval_f1': 0.8981481481481481,

### Performance Summary

| Model | Dataset | Accuracy | F1 |
|---|---|---|---|
| ANN | 1K | 0.83 | 0.83 |
| ANN | 25K | 0.86 | 0.86 |
| LSTM | 1K | 0.72 | 0.77 |
| LSTM | 25K | 0.83 | 0.83 |
| DistilBERT | 1K | 0.81 | 0.81 |
| DistilBERT | 25K | 0.86 | 0.89 |
| RoBERTa | 1K | 0.83 | 0.84 |
| RoBERTa | 25K | 0.89 | 0.91 |

1) ANN (TF-IDF + MLP)
Performs strongly because TF-IDF with unigram/bigram features captures sentiment cues directly (e.g., “great”, “not good”).
For short review texts, bag-of-words style features are often enough.
This is why ANN can beat LSTM, especially on the 1K dataset.

2) BiLSTM
Learns representations from scratch and needs more data/tuning.
In your setup, embeddings are randomly initialized, sequence length is truncated, and no pretrained embeddings are used.
This usually leads to lower performance than TF-IDF baselines on small/medium sentiment datasets.
On 25K it may improve, but often still trails optimized TF-IDF/transformers.

3) DistilBERT / RoBERTa
Best semantic understanding due to pretrained language knowledge.
Typically best generalization and F1 on sentiment tasks.
The shown metric (~0.88 acc / ~0.90 F1) is consistent with strong transformer behavior.

Dataset-size effects
1K data: simple models with strong priors (TF-IDF+ANN) are very competitive; LSTM is data-limited.
25K data: all models improve, but transformers usually gain the most and lead clearly.

Why ANN > LSTM here
Feature quality: ANN gets TF‑IDF unigram+bigrams, which are very strong for sentiment (e.g., “great”, “not good”).
Task type: Your texts are mostly short reviews, so word presence/phrases matter more than long sequence structure.
Data efficiency: TF‑IDF + MLP learns well with limited data; LSTM usually needs more data to beat sparse lexical features.
LSTM setup is harder: You train embeddings from scratch with small hidden size and regularization; this can underfit or learn noisy sequence patterns.

Why ANN is close to Transformer
Strong baseline: For binary sentiment, TF‑IDF often captures most of the signal already.
Domain simplicity: If labels are driven by explicit sentiment words, contextual modeling adds only a modest gain.
Fine-tuning constraints: With short training (few epochs, fixed max length, standard hyperparameters), transformers may not realize full advantage.
Net effect: Transformer still wins on larger data, but ANN remains very competitive because the problem is close to a bag‑of‑words task.